# 📊 CAT Data Lake — Row Count & Raw String Validator

**Purpose:** Source file (local) aur Trino (Silver) ka row count compare karna — total + event-wise. Raw string validation bhi.

| Source | Detail |
|--------|--------|
| **Source File** | Local `.bz2` file (CSV ya JSON) |
| **Destination** | Silver layer → Trino query |

### Checks Performed
| # | Check | Description |
|---|-------|-------------|
| 1 | **Total Row Count** | Source total rows vs Trino total rows |
| 2 | **Event-wise Row Count** | Per event type count match (MENO, MEOR, MEOC...) |
| 3 | **Missing / Extra Events** | Koi event source mein hai Trino mein nahi ya vice versa |
| 4 | **Raw String Validation** | Trino ki `raw_str` column = source row ka comma-joined string? |

### File Type Handling
| File Type | Event Type | Raw String |
|-----------|-----------|------------|
| CSV `.bz2` | Column index 3 (0-based) | Comma-joined row values |
| JSON `.bz2` | `type` key | JSON string of object |

---
## 📦 Cell 1 — Libraries

In [70]:
import bz2
import csv
import json
import urllib3
import pandas as pd
from collections import Counter
from IPython.display import display
from trino.dbapi import connect
from trino.auth import BasicAuthentication

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 200)

print('✅ Libraries loaded!')

✅ Libraries loaded!


---
## ⚙️ Cell 2 — Configuration (Sirf yahan change karo)

In [71]:
# ─────────────────────────────────────────────────────────────────
#  SOURCE FILE (local path)
# ─────────────────────────────────────────────────────────────────
SOURCE_FILE = r"/home/shariq/Downloads/CATFILE-VLVT/95707_VLCT_20260428_CATFLEX_OrderEvents_000003.csv.bz2"
# FILE_TYPE: 'csv' ya 'json' — filename se auto detect hoga, manually override bhi kar sakte ho
FILE_TYPE = None   # None = auto detect

# CSV specific
CSV_EVENT_TYPE_COL = 3   # 0-based index — event type kaunse column mein hai

# JSON specific
JSON_EVENT_TYPE_KEY = 'type'   # JSON mein event type ki key

# ─────────────────────────────────────────────────────────────────
#  TRINO CONNECTION
# ─────────────────────────────────────────────────────────────────
TRINO_CONFIG = {
    "host"    : "3.221.185.123",
    "port"    : 8443,
    "user"    : "shariq.mehmood",
    "password": "Prometheus-X!",
    "catalog" : "iceberg",
    "schema"  : "ignite_test"
}

TARGET_TABLE = 'iceberg.qa.silver_catevents'
WHERE_CLAUSE = "s3_path like '%s3://reportingfilesandbackup/CATManagementFiles/FlexTrade/VLCT-CATFLEX/CATReports/95707_VLCT_20260428_CATFLEX_OrderEvents_000003.csv.bz2%'"

# ─────────────────────────────────────────────────────────────────
#  RAW STRING VALIDATION
# ─────────────────────────────────────────────────────────────────
# Kitni rows raw_str validate karni hain (sample)
# None = saari rows validate karo (slow ho sakta hai)
RAW_STR_SAMPLE_SIZE = 1000

# ── Run to check null values──
SCHEMA_FILE_PATH = r"/home/shariq/Downloads/data_configurator_public_cat_event_schema.csv"
NULL_SAMPLE_SIZE = 5000   # source aur Trino dono se same size

print(f'✅ Source File  : {SOURCE_FILE}')
print(f'✅ Target Table : {TARGET_TABLE}')
print(f'✅ Null check Sample   : {NULL_SAMPLE_SIZE} rows')
print(f'✅ Rawstring Sample   : {RAW_STR_SAMPLE_SIZE} rows')

✅ Source File  : /home/shariq/Downloads/CATFILE-VLVT/95707_VLCT_20260428_CATFLEX_OrderEvents_000003.csv.bz2
✅ Target Table : iceberg.qa.silver_catevents
✅ Null check Sample   : 5000 rows
✅ Rawstring Sample   : 1000 rows


---
## 📂 Cell 3 — Source File Read karo

In [72]:
def detect_file_type(path):
    """Filename se file type detect karo."""
    name = path.lower()
    if '_json.bz2' in name or name.endswith('.json.bz2') or name.endswith('.json'):
        return 'json'
    return 'csv'


def read_source_file(path, file_type=None):
    """
    Source file read karo (.bz2 ya plain).
    Returns:
        file_type    : 'csv' ya 'json'
        rows         : list of rows
                       CSV  → list of lists (values)
                       JSON → list of dicts
        event_counts : Counter {event_type: count}
        raw_strings  : list of raw strings (comma-joined for CSV, json str for JSON)
    """
    ftype = file_type or detect_file_type(path)
    is_bz2 = path.endswith('.bz2')

    open_fn = bz2.open if is_bz2 else open

    rows        = []
    raw_strings = []
    event_counts = Counter()

    print(f'⏳ Reading {ftype.upper()} file ({'bz2 compressed' if is_bz2 else 'plain'})...')

    if ftype == 'csv':
        with open_fn(path, 'rt', encoding='utf-8') as f:
            reader = csv.reader(f)
            for row in reader:
                if not row:   # empty line skip
                    continue
                rows.append(row)
                raw_strings.append(','.join(row))
                if len(row) > CSV_EVENT_TYPE_COL:
                    event_counts[row[CSV_EVENT_TYPE_COL]] += 1
                else:
                    event_counts['__UNKNOWN__'] += 1

    elif ftype == 'json':
        with open_fn(path, 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                rows.append(obj)
                raw_strings.append(json.dumps(obj, separators=(',', ':')))
                event_type = obj.get(JSON_EVENT_TYPE_KEY, '__UNKNOWN__')
                event_counts[event_type] += 1

    print(f'\n✅ Source file read complete!')
    print(f'   📊 Total rows        : {len(rows)}')
    print(f'   🗂️  Unique event types : {len(event_counts)}')
    print(f'\n   Event-wise breakdown (SOURCE):')
    for evt, cnt in sorted(event_counts.items()):
        print(f'      {evt:<12} : {cnt}')

    return ftype, rows, event_counts, raw_strings


file_type, source_rows, source_event_counts, source_raw_strings = read_source_file(SOURCE_FILE, FILE_TYPE)

⏳ Reading CSV file (bz2 compressed)...

✅ Source file read complete!
   📊 Total rows        : 11138
   🗂️  Unique event types : 12

   Event-wise breakdown (SOURCE):
      MLCR         : 12
      MLMR         : 15
      MLOA         : 24
      MLOC         : 10
      MLOM         : 15
      MLOR         : 24
      MOCR         : 357
      MOMR         : 3109
      MOOA         : 2030
      MOOC         : 382
      MOOM         : 3103
      MOOR         : 2057


---
## 🔌 Cell 4 — Trino se Row Count Fetch karo

In [73]:
def get_trino_connection():
    auth = BasicAuthentication(TRINO_CONFIG['user'], TRINO_CONFIG['password'])
    return connect(
        host        = TRINO_CONFIG['host'],
        port        = TRINO_CONFIG['port'],
        user        = TRINO_CONFIG['user'],
        auth        = auth,
        http_scheme = 'https',
        verify      = False,
        catalog     = TRINO_CONFIG['catalog'],
        schema      = TRINO_CONFIG['schema']
    )
 
 
def fetch_trino_counts():
    """
    Trino se event-wise row count aur total count fetch karo.
    Returns: (total_count, event_counts_dict)
    """
    conn = get_trino_connection()
    cur  = conn.cursor()
 
    # Event-wise count
    query_event = f"""
        SELECT "type", COUNT(*) as cnt
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        GROUP BY "type"
        ORDER BY "type"
    """
    print('⏳ Trino se event-wise count fetch ho raha hai...')
    cur.execute(query_event)
    trino_event_counts = Counter({row[0]: row[1] for row in cur.fetchall()})
 
    total = sum(trino_event_counts.values())
 
    print(f'\n✅ Trino count fetch complete!')
    print(f'   📊 Total rows        : {total}')
    print(f'   🗂️  Unique event types : {len(trino_event_counts)}')
    print(f'\n   Event-wise breakdown (TRINO):')
    for evt, cnt in sorted(trino_event_counts.items()):
        print(f'      {evt:<12} : {cnt}')
 
    return total, trino_event_counts
 
 
trino_total, trino_event_counts = fetch_trino_counts()

⏳ Trino se event-wise count fetch ho raha hai...

✅ Trino count fetch complete!
   📊 Total rows        : 11138
   🗂️  Unique event types : 12

   Event-wise breakdown (TRINO):
      MLCR         : 12
      MLMR         : 15
      MLOA         : 24
      MLOC         : 10
      MLOM         : 15
      MLOR         : 24
      MOCR         : 357
      MOMR         : 3109
      MOOA         : 2030
      MOOC         : 382
      MOOM         : 3103
      MOOR         : 2057


---
## 🔢 Cell 5 — Row Count Comparison

In [74]:
source_total = len(source_rows)

print('=' * 70)
print('📊 ROW COUNT COMPARISON')
print('=' * 70)

# ── Total count ──
total_match = source_total == trino_total
total_icon  = '✅' if total_match else '❌'
print(f'\n{total_icon} TOTAL ROWS')
print(f'   Source : {source_total}')
print(f'   Trino  : {trino_total}')
if not total_match:
    diff = trino_total - source_total
    print(f'   ❌ Difference : {diff:+d} ({"extra in Trino" if diff > 0 else "missing in Trino"})')

# ── Event-wise count ──
print(f'\n{"-" * 70}')
print(f'  {"EVENT TYPE":<14} | {"SOURCE":>8} | {"TRINO":>8} | {"DIFF":>8} | STATUS')
print(f'{"-" * 70}')

all_events    = sorted(set(source_event_counts.keys()) | set(trino_event_counts.keys()))
event_results = []
all_match     = True

for evt in all_events:
    src_cnt   = source_event_counts.get(evt, 0)
    trn_cnt   = trino_event_counts.get(evt, 0)
    diff      = trn_cnt - src_cnt
    match     = src_cnt == trn_cnt
    if not match:
        all_match = False

    # Status label
    if match:
        status = '✅ MATCH'
    elif src_cnt == 0:
        status = '🔵 EXTRA IN TRINO'
    elif trn_cnt == 0:
        status = '🔴 MISSING IN TRINO'
    else:
        status = f'❌ MISMATCH ({diff:+d})'

    print(f'  {evt:<14} | {src_cnt:>8} | {trn_cnt:>8} | {diff:>+8} | {status}')
    event_results.append({'event_type': evt, 'source_count': src_cnt,
                          'trino_count': trn_cnt, 'diff': diff, 'status': status})

print(f'{"-" * 70}')
print(f'  {"TOTAL":<14} | {source_total:>8} | {trino_total:>8} | {trino_total - source_total:>+8} | {"✅ MATCH" if total_match else "❌ MISMATCH"}')
print(f'=' * 70)

if all_match and total_match:
    print('\n✅ ALL COUNTS MATCH — Source aur Trino mein row count bilkul same hai!')
else:
    print('\n❌ COUNT MISMATCH FOUND — Details upar table mein dekho!')

row_count_df = pd.DataFrame(event_results)

📊 ROW COUNT COMPARISON

✅ TOTAL ROWS
   Source : 11138
   Trino  : 11138

----------------------------------------------------------------------
  EVENT TYPE     |   SOURCE |    TRINO |     DIFF | STATUS
----------------------------------------------------------------------
  MLCR           |       12 |       12 |       +0 | ✅ MATCH
  MLMR           |       15 |       15 |       +0 | ✅ MATCH
  MLOA           |       24 |       24 |       +0 | ✅ MATCH
  MLOC           |       10 |       10 |       +0 | ✅ MATCH
  MLOM           |       15 |       15 |       +0 | ✅ MATCH
  MLOR           |       24 |       24 |       +0 | ✅ MATCH
  MOCR           |      357 |      357 |       +0 | ✅ MATCH
  MOMR           |     3109 |     3109 |       +0 | ✅ MATCH
  MOOA           |     2030 |     2030 |       +0 | ✅ MATCH
  MOOC           |      382 |      382 |       +0 | ✅ MATCH
  MOOM           |     3103 |     3103 |       +0 | ✅ MATCH
  MOOR           |     2057 |     2057 |       +0 | ✅ MATCH
-----

---
## 🧵 Cell 6 — Raw String Validation

In [75]:
def fetch_trino_rawstrings(sample_size=None):
    """
    Trino se raw_str column fetch karo.
    sample_size = None means fetch all.
    Returns list of raw_str values from Trino.
    """
    conn = get_trino_connection()
    cur  = conn.cursor()

    limit_clause = f'LIMIT {sample_size}' if sample_size else ''
    query = f"""
        SELECT raw_str
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        {limit_clause}
    """
    print(f'⏳ Trino se raw_str fetch ho raha hai ({sample_size if sample_size else "all"} rows)...')
    cur.execute(query)
    results = [row[0] for row in cur.fetchall()]
    print(f'✅ Fetched {len(results)} raw_str values from Trino')
    return results


def validate_raw_strings(source_raws, trino_raws, file_type):
    """
    Source raw strings vs Trino raw_str compare karo.
    Note: Order match guarantee nahi hoti (Trino unordered) —
    isliye set-based comparison + sample spot check dono karenge.
    """
    print('\n' + '=' * 70)
    print('🧵 RAW STRING VALIDATION')
    print('=' * 70)
    print(f'   File type     : {file_type.upper()}')
    print(f'   Source rows   : {len(source_raws)}')
    print(f'   Trino rows    : {len(trino_raws)}')

    # ── Null check in Trino raw_str ──
    trino_nulls = sum(1 for r in trino_raws if not r or str(r).strip() in {'', 'None', 'null', 'nan'})
    print(f'\n   🔍 Null raw_str in Trino : {trino_nulls}')
    if trino_nulls > 0:
        print(f'   ❌ {trino_nulls} rows mein raw_str NULL hai — ingestion issue!')
    else:
        print(f'   ✅ Koi null raw_str nahi')

    # ── Set-based match ──
    source_set = set(source_raws)
    trino_set  = set(str(r) for r in trino_raws if r)

    in_source_not_trino = source_set - trino_set
    in_trino_not_source = trino_set - source_set

    print(f'\n   📊 Set-based comparison (unique raw strings):')
    print(f'      Source unique  : {len(source_set)}')
    print(f'      Trino unique   : {len(trino_set)}')

    if not in_source_not_trino and not in_trino_not_source:
        print(f'\n   ✅ PASS — Sab raw strings match karte hain!')
    else:
        if in_source_not_trino:
            print(f'\n   🔴 Source mein hai, Trino mein NAHI ({len(in_source_not_trino)}):')  
            for s in list(in_source_not_trino)[:5]:
                print(f'      → {s[:120]}')
            if len(in_source_not_trino) > 5:
                print(f'      ... aur {len(in_source_not_trino) - 5} aur')

        if in_trino_not_source:
            print(f'\n   🔵 Trino mein hai, Source mein NAHI ({len(in_trino_not_source)}):')
            for s in list(in_trino_not_source)[:5]:
                print(f'      → {str(s)[:120]}')
            if len(in_trino_not_source) > 5:
                print(f'      ... aur {len(in_trino_not_source) - 5} aur')

    # ── Sample spot check ──
    print(f'\n   🔎 Sample Spot Check (first 5 source rows vs Trino):')
    print(f'   {"─" * 60}')
    for i, src_raw in enumerate(source_raws[:5]):
        found = str(src_raw) in trino_set
        icon  = '✅' if found else '❌'
        print(f'   {icon} Row {i+1}: {str(src_raw)[:100]}...')
        if not found:
            print(f'      ↳ NOT FOUND in Trino raw_str')

    return {
        'trino_nulls'         : trino_nulls,
        'in_source_not_trino' : len(in_source_not_trino),
        'in_trino_not_source' : len(in_trino_not_source),
        'raw_str_pass'        : trino_nulls == 0 and not in_source_not_trino and not in_trino_not_source
    }


trino_raw_strings = fetch_trino_rawstrings(RAW_STR_SAMPLE_SIZE)
raw_str_results   = validate_raw_strings(
    source_raws = source_raw_strings[:RAW_STR_SAMPLE_SIZE] if RAW_STR_SAMPLE_SIZE else source_raw_strings,
    trino_raws  = trino_raw_strings,
    file_type   = file_type
)

⏳ Trino se raw_str fetch ho raha hai (1000 rows)...
✅ Fetched 1000 raw_str values from Trino

🧵 RAW STRING VALIDATION
   File type     : CSV
   Source rows   : 1000
   Trino rows    : 1000

   🔍 Null raw_str in Trino : 0
   ✅ Koi null raw_str nahi

   📊 Set-based comparison (unique raw strings):
      Source unique  : 1000
      Trino unique   : 1000

   ✅ PASS — Sab raw strings match karte hain!

   🔎 Sample Spot Check (first 5 source rows vs Trino):
   ────────────────────────────────────────────────────────────
   ✅ Row 1: NEW,,20260428_OPT_MOOA_212604280068544,MOOA,VLCT,20260428T000000,USH59570007766,AAOI  260501C0015000...
   ✅ Row 2: NEW,,20260428_OPT_MOOA_212604280097757,MOOA,VLCT,20260428T000000,USH59570010951,AAOI  260501C0015000...
   ✅ Row 3: NEW,,20260428_OPT_MOOA_212604280100238,MOOA,VLCT,20260428T000000,USH59570011108,AAOI  260501C0015000...
   ✅ Row 4: NEW,,20260428_OPT_MOOA_212604280100526,MOOA,VLCT,20260428T000000,USH59570011240,AAOI  260501C0015000...
   ✅ Row 5: NEW,

---
## 🧵 Cell 9 — Raw String: Trino Fetch + Compare
Trino se actual `raw_str` values fetch karo aur source se compare karo.
Agar trailing comma mismatch ho toh `STRIP_TRAILING` mode auto-detect karta hai.

In [76]:
def fetch_trino_rawstr_with_firmroeid(sample_size=50):
    """
    Trino se firmroeid + raw_str fetch karo.
    """
    conn = get_trino_connection()
    cur  = conn.cursor()
    query = f"""
        SELECT firmroeid, raw_str
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        LIMIT {sample_size}
    """
    print(f'⏳ Trino se firmroeid + raw_str fetch ho raha hai ({sample_size} rows)...')
    cur.execute(query)
    rows = cur.fetchall()
    df   = pd.DataFrame(rows, columns=['firmroeid', 'raw_str'])
    df['firmroeid'] = df['firmroeid'].astype(str).str.strip()
    print(f'✅ {len(df)} rows fetched')
    return df


def build_source_rawstr_with_firmroeid(source_rows, file_type):
    """
    Source rows se firmroeid + raw_str DataFrame banao.
    firmROEID position index 2 (positionkey 3) — har event mein same.
    """
    records = []
    if file_type == 'csv':
        for i, row in enumerate(source_rows):
            firmroeid = row[2].strip() if len(row) > 2 else ''
            raw_str   = ','.join(row)
            records.append({
                'source_row_num': i + 1,
                'firmroeid'     : firmroeid,
                'raw_str_src'   : raw_str
            })
    elif file_type == 'json':
        for i, obj in enumerate(source_rows):
            firmroeid = str(obj.get('firmROEID', obj.get('firmroeid', ''))).strip()
            raw_str   = json.dumps(obj, separators=(',', ':'))
            records.append({
                'source_row_num': i + 1,
                'firmroeid'     : firmroeid,
                'raw_str_src'   : raw_str
            })
    return pd.DataFrame(records)


def compare_rawstrings_with_join(trino_raw_df, source_raw_df):
    """
    firmROEID se join karo phir raw_str compare karo.
    """
    print('\n' + '=' * 90)
    print('🧵 RAW STRING COMPARISON — firmROEID JOIN')
    print('=' * 90)

    # Join on firmroeid
    merged = source_raw_df.merge(trino_raw_df, on='firmroeid', suffixes=('', '_trino'))

    print(f'   Source rows  : {len(source_raw_df)}')
    print(f'   Trino rows   : {len(trino_raw_df)}')
    print(f'   Joined rows  : {len(merged)}')

    unmatched_src   = set(source_raw_df['firmroeid']) - set(trino_raw_df['firmroeid'])
    unmatched_trino = set(trino_raw_df['firmroeid']) - set(source_raw_df['firmroeid'])
    if unmatched_src:
        print(f'   ⚠️  Source firmROEIDs not in Trino  : {len(unmatched_src)}')
    if unmatched_trino:
        print(f'   ⚠️  Trino firmROEIDs not in Source  : {len(unmatched_trino)}')

    # NULL check in Trino raw_str
    trino_nulls = merged['raw_str'].isna() | merged['raw_str'].astype(str).str.strip().isin(
                  ['', 'None', 'null', 'nan'])
    null_count  = int(trino_nulls.sum())
    print(f'\n   🔍 NULL raw_str in Trino : {null_count} {"✅" if null_count == 0 else "❌"}')

    # Compare raw strings
    merged['raw_match'] = merged['raw_str_src'].astype(str).str.strip() == \
                           merged['raw_str'].astype(str).str.strip()

    match_count    = int(merged['raw_match'].sum())
    mismatch_count = int((~merged['raw_match']).sum())

    # ── Per row output ──
    print(f'\n{"-" * 90}')
    print(f'  {"ROW":>4} | {"firmROEID":<45} | {"STATUS"}')
    print(f'{"-" * 90}')

    for _, row in merged.iterrows():
        icon = '✅ MATCH' if row['raw_match'] else '❌ MISMATCH'
        print(f'  {int(row["source_row_num"]):>4} | {str(row["firmroeid"]):<45} | {icon}')
        if not row['raw_match']:
            print(f'       SRC  : {str(row["raw_str_src"])[:100]}')
            print(f'       TRINO: {str(row["raw_str"])[:100]}')

    print(f'{"-" * 90}')
    print(f'  ✅ MATCH    : {match_count}')
    print(f'  ❌ MISMATCH : {mismatch_count}')
    print('=' * 90)

    return merged, match_count, mismatch_count


# ── Run ──
source_raw_df     = build_source_rawstr_with_firmroeid(source_rows[:RAW_STR_SAMPLE_SIZE], file_type)
trino_raw_df      = fetch_trino_rawstr_with_firmroeid(RAW_STR_SAMPLE_SIZE)
merged_raw_df, raw_match, raw_mismatch = compare_rawstrings_with_join(trino_raw_df, source_raw_df)


⏳ Trino se firmroeid + raw_str fetch ho raha hai (1000 rows)...
✅ 1000 rows fetched

🧵 RAW STRING COMPARISON — firmROEID JOIN
   Source rows  : 1000
   Trino rows   : 1000
   Joined rows  : 1000

   🔍 NULL raw_str in Trino : 0 ✅

------------------------------------------------------------------------------------------
   ROW | firmROEID                                     | STATUS
------------------------------------------------------------------------------------------
     1 | 20260428_OPT_MOOA_212604280068544             | ✅ MATCH
     2 | 20260428_OPT_MOOA_212604280097757             | ✅ MATCH
     3 | 20260428_OPT_MOOA_212604280100238             | ✅ MATCH
     4 | 20260428_OPT_MOOA_212604280100526             | ✅ MATCH
     5 | 20260428_OPT_MOOA_212604280059363             | ✅ MATCH
     6 | 20260428_OPT_MOOA_212604280059129             | ✅ MATCH
     7 | 20260428_OPT_MOOA_212604280060064             | ✅ MATCH
     8 | 20260428_OPT_MOOA_212604280114970             | ✅ MATCH
    

---
## 📤 Cell — Script Raw Strings Export (Top 50)
Script ki banai hui top 50 raw strings print karo — Excel mein 3rd column k liye.

In [ ]:
# Script ki banai hui raw strings — top 50
# Ye wahi strings hain jo Cell 9 mein Trino se compare ki gayi hain

print('📤 SCRIPT RAW STRINGS — TOP 50')
print('(Excel mein 3rd column mein paste karo)')
print('=' * 80)

for i, raw in enumerate(source_raw_strings[:50], 1):
    print(f'{raw}')


---
## 🔬 Cell 10 — Null Validation: True NULL ya Source Empty?
Trino mein jo columns NULL dikh rahe hain — kya woh source mein bhi empty the?
Agar source mein value thi aur Trino mein NULL hai → **ingestion bug**.
Agar source mein bhi empty tha → **expected NULL**.

In [77]:
import pandas as pd

def fetch_trino_full_data():
    """
    Trino se saara data fetch karo — firmroeid bhi include.
    """
    conn = get_trino_connection()
    cur  = conn.cursor()
    query = f"""
        SELECT *
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        LIMIT {NULL_SAMPLE_SIZE}
    """
    print(f'⏳ Trino se data fetch ho raha hai ({NULL_SAMPLE_SIZE} rows)...')
    cur.execute(query)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    df.columns = df.columns.str.lower()
    print(f'✅ {len(df)} rows, {len(df.columns)} columns fetched')
    return df


def build_source_df(source_rows, file_type, schema_file):
    """
    Source rows ko schema positionkey se named DataFrame banao.
    firmroeid position 3 (index 2) — har event mein same.
    """
    schema = pd.read_csv(schema_file)
    schema.columns = schema.columns.str.strip()
    schema['fieldname'] = schema['fieldname'].str.strip().str.lower()
    schema['eventtype'] = schema['eventtype'].str.strip()

    event_pos_map = {}
    for evt, grp in schema.groupby('eventtype'):
        event_pos_map[evt] = {int(r['positionkey']) - 1: r['fieldname']
                              for _, r in grp.iterrows()}

    records = []
    if file_type == 'csv':
        for i, row in enumerate(source_rows):
            if len(row) <= CSV_EVENT_TYPE_COL:
                continue
            evt     = row[CSV_EVENT_TYPE_COL]
            pos_map = event_pos_map.get(evt, {})
            record  = {'source_row_num': i + 1, 'type': evt}
            for pos_idx, fieldname in pos_map.items():
                record[fieldname] = row[pos_idx].strip() if pos_idx < len(row) else ''
            # raw_str
            record['raw_str'] = ','.join(row)
            records.append(record)

    elif file_type == 'json':
        for i, obj in enumerate(source_rows):
            record = {'source_row_num': i + 1}
            record.update({k.lower(): str(v) for k, v in obj.items()})
            record['raw_str'] = json.dumps(obj, separators=(',', ':'))
            records.append(record)

    df = pd.DataFrame(records).fillna('')
    return df


def validate_nulls_with_join(trino_df, source_df):
    """
    firmroeid se join karo — phir:
    1. raw_str match check
    2. Column-wise null validation (true null vs false null)
    """
    print('\n' + '=' * 100)
    print('🔬 NULL VALIDATION — firmROEID JOIN (True NULL vs Ingestion Bug)')
    print('=' * 100)

    # ── firmROEID se join ──
    if 'firmroeid' not in trino_df.columns:
        print('❌ firmroeid column Trino mein nahi mili!')
        return pd.DataFrame(), pd.DataFrame()
    if 'firmroeid' not in source_df.columns:
        print('❌ firmroeid column source mein nahi mili!')
        return pd.DataFrame(), pd.DataFrame()

    trino_df  = trino_df.copy()
    source_df = source_df.copy()
    trino_df['firmroeid']  = trino_df['firmroeid'].astype(str).str.strip()
    source_df['firmroeid'] = source_df['firmroeid'].astype(str).str.strip()

    merged = source_df.merge(trino_df, on='firmroeid', suffixes=('_src', '_trino'))
    print(f'\n   Source rows      : {len(source_df)}')
    print(f'   Trino rows       : {len(trino_df)}')
    print(f'   Joined rows      : {len(merged)}')

    unmatched = len(source_df) - len(merged)
    if unmatched > 0:
        print(f'   ⚠️  Unmatched rows : {unmatched} (firmroeid Trino mein nahi mili)')

    # ── Raw string match check ──
    print(f'\n{"-" * 60}')
    print('🧵 RAW STRING CHECK (per joined row):')

    if 'raw_str_src' in merged.columns and 'raw_str_trino' in merged.columns:
        merged['raw_match'] = merged['raw_str_src'].astype(str).str.strip() == \
                               merged['raw_str_trino'].astype(str).str.strip()
        raw_pass  = merged['raw_match'].sum()
        raw_fail  = (~merged['raw_match']).sum()
        print(f'   ✅ Match   : {raw_pass}/{len(merged)}')
        if raw_fail > 0:
            print(f'   ❌ Mismatch: {raw_fail}')
            mismatch_rows = merged[~merged['raw_match']][['firmroeid','raw_str_src','raw_str_trino']]
            for _, r in mismatch_rows.head(5).iterrows():
                print(f'   firmroeid: {r["firmroeid"]}')
                print(f'     SRC  : {str(r["raw_str_src"])[:100]}')
                print(f'     TRINO: {str(r["raw_str_trino"])[:100]}')
    else:
        print('   ⚠️  raw_str columns nahi mili — skip')

    # ── Null validation ──
    skip_cols = {'raw_str_src', 'raw_str_trino', 'raw_match', 's3_path',
                 'ingestedfordate', 'keydate', 'boothid', 'sourcefile',
                 'submitterid', 'vendor', 'source_row_num', 'type_src',
                 'type_trino', 'firmroeid'}

    # Schema columns — _src suffix wale
    src_cols = [c for c in merged.columns
                if c.endswith('_src') and c not in skip_cols]
    schema_cols = [c.replace('_src', '') for c in src_cols]

    print(f'\n{"-" * 100}')
    print('🔬 COLUMN NULL VALIDATION:')
    print('  Legend: ✅ TRUE NULL = source bhi empty | ❌ FALSE NULL = source mein value thi | ⏭️ SKIP = no nulls')
    print(f'\n  {"COLUMN":<35} | {"TRINO NULL":>10} | {"SRC EMPTY":>10} | {"FALSE NULL":>10} | STATUS')
    print(f'  {"-" * 100}')

    results          = []
    false_null_detail = []

    for col in sorted(schema_cols):
        src_col   = col + '_src'
        trino_col = col + '_trino'

        if src_col not in merged.columns or trino_col not in merged.columns:
            continue

        t_vals = merged[trino_col]
        s_vals = merged[src_col]

        trino_null_mask   = t_vals.isna() | t_vals.astype(str).str.strip().isin(
                            ['', 'None', 'null', 'nan', 'NaN'])
        source_empty_mask = s_vals.astype(str).str.strip() == ''
        false_null_mask   = trino_null_mask & ~source_empty_mask

        trino_null_count   = int(trino_null_mask.sum())
        source_empty_count = int(source_empty_mask.sum())
        false_null_count   = int(false_null_mask.sum())

        if trino_null_count == 0:
            status = '⏭️  SKIP'
        elif false_null_count > 0:
            status = '❌ FALSE NULL'

            false_rows = merged[false_null_mask][['firmroeid', 'source_row_num',
                                                   'type_src', src_col]].copy()
            false_rows.columns = ['firmroeid', 'source_row_num', 'event_type', 'source_value']

            if false_null_count < 10:
                # Row level detail
                for _, r in false_rows.iterrows():
                    false_null_detail.append({
                        'column'      : col,
                        'row_number'  : int(r['source_row_num']),
                        'firmroeid'   : r['firmroeid'],
                        'event_type'  : r['event_type'],
                        'source_value': str(r['source_value'])[:50]
                    })
            else:
                # Event type distribution
                evt_dist = false_rows['event_type'].value_counts().to_dict()
                false_null_detail.append({
                    'column'      : col,
                    'row_number'  : f'{false_null_count} rows',
                    'firmroeid'   : 'multiple',
                    'event_type'  : str(evt_dist),
                    'source_value': 'multiple'
                })
        else:
            status = '✅ TRUE NULL'

        if trino_null_count > 0:
            print(f'  {col:<35} | {trino_null_count:>10} | {source_empty_count:>10} | '
                  f'{false_null_count:>10} | {status}')

        results.append({'column': col, 'trino_null': trino_null_count,
                        'source_empty': source_empty_count,
                        'false_null': false_null_count, 'status': status})

    print(f'  {"-" * 100}')

    # ── False null detail table ──
    if false_null_detail:
        print(f'\n❌ FALSE NULL DETAIL:\n')
        print(f'  {"COLUMN":<35} | {"ROW":>6} | {"firmROEID":<45} | {"EVENT":>6} | SOURCE VALUE')
        print(f'  {"-" * 120}')
        for r in false_null_detail:
            print(f'  {str(r["column"]):<35} | {str(r["row_number"]):>6} | '
                  f'{str(r["firmroeid"]):<45} | {str(r["event_type"]):>6} | {str(r["source_value"])[:30]}')
    else:
        print('\n✅ Koi FALSE NULL nahi — sab nulls source mein bhi empty the!')

    return pd.DataFrame(results), pd.DataFrame(false_null_detail) if false_null_detail else pd.DataFrame()


# ── Run ──
trino_full_df  = fetch_trino_full_data()
source_df      = build_source_df(source_rows[:NULL_SAMPLE_SIZE], file_type, SCHEMA_FILE_PATH)
null_results_df, false_null_df = validate_nulls_with_join(trino_full_df, source_df)


⏳ Trino se data fetch ho raha hai (5000 rows)...
✅ 5000 rows, 121 columns fetched

🔬 NULL VALIDATION — firmROEID JOIN (True NULL vs Ingestion Bug)

   Source rows      : 5000
   Trino rows       : 5000
   Joined rows      : 5000

------------------------------------------------------------
🧵 RAW STRING CHECK (per joined row):
   ✅ Match   : 5000/5000

----------------------------------------------------------------------------------------------------
🔬 COLUMN NULL VALIDATION:
  Legend: ✅ TRUE NULL = source bhi empty | ❌ FALSE NULL = source mein value thi | ⏭️ SKIP = no nulls

  COLUMN                              | TRINO NULL |  SRC EMPTY | FALSE NULL | STATUS
  ----------------------------------------------------------------------------------------------------
  depttype                            |       2970 |       2970 |          0 | ✅ TRUE NULL
  destination                         |       2054 |       2054 |          0 | ✅ TRUE NULL
  destinationtype                     |       

---
## 📋 Cell 7 — Final Summary

In [61]:
print('\n' + '=' * 70)
print('📋 FINAL SUMMARY')
print('=' * 70)

checks = [
    ('Total Row Count',    '✅ PASS' if total_match else '❌ FAIL',
     f'Source={source_total}, Trino={trino_total}'),

    ('Event-wise Count',   '✅ PASS' if all_match else '❌ FAIL',
     f'{sum(1 for r in event_results if "MATCH" in r["status"])}/{len(event_results)} events match'),

    ('Raw String NULLs',   '✅ PASS' if raw_str_results['trino_nulls'] == 0 else '❌ FAIL',
     f"{raw_str_results['trino_nulls']} null raw_str in Trino"),

    ('Raw String Match',   '✅ PASS' if raw_str_results['raw_str_pass'] else '❌ FAIL',
     f"Source-only: {raw_str_results['in_source_not_trino']}, Trino-only: {raw_str_results['in_trino_not_source']}"),
]

print(f'\n  {"CHECK":<25} | {"STATUS":<12} | DETAIL')
print(f'  {"-" * 65}')
for check, status, detail in checks:
    print(f'  {check:<25} | {status:<12} | {detail}')

print(f'\n  Source File : {SOURCE_FILE}')
print(f'  Table       : {TARGET_TABLE}')
print('=' * 70)

overall_pass = all(status == '✅ PASS' for _, status, _ in checks)
if overall_pass:
    print('\n🎉 ALL CHECKS PASSED!')
else:
    print('\n❌ SOME CHECKS FAILED — Details upar dekho!')


📋 FINAL SUMMARY

  CHECK                     | STATUS       | DETAIL
  -----------------------------------------------------------------
  Total Row Count           | ✅ PASS       | Source=455516, Trino=455516
  Event-wise Count          | ✅ PASS       | 10/10 events match
  Raw String NULLs          | ✅ PASS       | 0 null raw_str in Trino
  Raw String Match          | ✅ PASS       | Source-only: 0, Trino-only: 0

  Source File : /home/shariq/Downloads/CATFILE-VLVT/96547_VCLT_20260427_OrderEvents_000001.csv.bz2
  Table       : iceberg.qa.silver_catevents

🎉 ALL CHECKS PASSED!


---
## 🔎 Cell 8 — Debug: Raw String Mismatch Detail (Optional)

In [ ]:
# Agar raw string mismatch aaya ho toh yahan specific rows inspect karo
# Konsi source row Trino mein match nahi hui

trino_set_full = set(str(r) for r in trino_raw_strings if r)

mismatch_rows = []
check_raws = source_raw_strings[:RAW_STR_SAMPLE_SIZE] if RAW_STR_SAMPLE_SIZE else source_raw_strings

for i, raw in enumerate(check_raws):
    if str(raw) not in trino_set_full:
        mismatch_rows.append({'source_row_index': i + 1, 'source_raw': str(raw)[:200]})

if mismatch_rows:
    print(f'❌ Raw string mismatches found: {len(mismatch_rows)}')
    display(pd.DataFrame(mismatch_rows))
else:
    print('✅ Koi raw string mismatch nahi — sab checked rows Trino mein hain!')